In [ ]:
"""
In this notebook we will create a prompt JSON file for the dataset, which contains all the possible augmentations
"""

In [1]:
import sys, os
PROJECT_ROOT = "/hpc/dctrl/ks723/Data_augmentation"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


In [ ]:
"""This creates a json file that contains information about source and target files and blank prompt for all training image pairs
input is just the two folders containing the input and output images, change accordingly"""

Here, our goal is to choose different datasets to train the augmentation model. For all datasets, the main idea is to group by replicates, and set source and target filenames between each possible replicate. 

So for ex there are 3 replicate images - A, B and C
Possible pairs are AB, AC and BC; where first letter indicates source and second depicts target.

In [ ]:
"""
First with my experimental dataset from my previous paper
"""

In [ ]:
# For mapping between replicates

# Logic 
# Files are labelled like {Fixed}_{Imagenumber}_{replicate_number}_{augmentation_rotangle}.TIF 
# Fixed is either Fixed or empty, image directly starts from Image number
# Most images have 2 replicates, in these cases the pairs will be first and second replicate
# There are some images with only one replicate, these will be discarded
# Some images have more than 2 replicate, like upto 6 I think, in these cases we will have Nc2 pairs, i.e. all combinations of replicate pairs

from itertools import combinations
import os
from Data_augmentation.local_config import EXP_IMAGES_FOLDER, BASE_FOLDER
import json
import natsort

base_folder=BASE_FOLDER # prompt_experiments_augmentation.json will be saved here

folder=EXP_IMAGES_FOLDER

# save the unique image numbers and their corresponding replicate numbers
# Key: (image_number, rotation_angle), Value: list of replicate numbers
image_replicates = {}
for filename in natsort.natsorted(os.listdir(folder)):
    if filename.endswith('.TIF'):
        parts = filename.split('_')
        
        if parts[0] == 'Fixed':
            image_number = parts[0] + '_' + parts[1]
            replicate_number = parts[2]
            rotation_angle = parts[3].replace('.TIF', '')  # e.g., "rot356.4" or "rot338.400000000003"
        else:
            image_number = parts[0]
            replicate_number = parts[1]
            rotation_angle = parts[2].replace('.TIF', '')  # e.g., "rot356.4" or "rot338.400000000003"
        
        # Group by image number AND rotation angle
        key = (image_number, rotation_angle)
        
        if key not in image_replicates:
            image_replicates[key] = []
        image_replicates[key].append(replicate_number)

# discard images with less than 2 replicates
image_replicates = {key: reps for key, reps in image_replicates.items() if len(reps) >= 2}

# Generate the prompt.json file with the NC2 combinations of replicates

json_path = os.path.join(base_folder, "prompt_experiments_augmentation.json")
with open(json_path, "w") as f:
    for (image_number, rotation_angle), replicate_numbers in image_replicates.items():
        # get all combinations of replicate pairs
        for rep1, rep2 in combinations(replicate_numbers, 2):
            # reconstruct filenames with rotation angle (preserves exact string from original filename)
            src_filename = f"{image_number}_{rep1}_{rotation_angle}.TIF"
            tgt_filename = f"{image_number}_{rep2}_{rotation_angle}.TIF"
            
            entry = {
                "source": src_filename,
                "target": tgt_filename,
                "prompt": ""  # Empty text prompt
            }
            f.write(json.dumps(entry) + "\n")  # Write each entry as a new line

# count number of entries
number_of_entries = sum(1 for line in open(json_path))  
print(f"Generated {json_path} with {number_of_entries} entries.")

# perform a sanity check that all files exist
with open(json_path, "r") as f:
    for line in f:
        entry = json.loads(line)
        src_path = os.path.join(folder, entry["source"])
        tgt_path = os.path.join(folder, entry["target"])
        assert os.path.exists(src_path), f"Source file {src_path} does not exist!"
        assert os.path.exists(tgt_path), f"Target file {tgt_path} does not exist!"
print("All source and target files exist.")


In [ ]:
"""
Create a backup file for safety 
"""

Add folder paths to existing entries and combine with new mutant dataset

In [ ]:
# Step 1: Load existing entries and add folder path, then create backup
import json
import os
import shutil
from local_config import EXP_IMAGES_FOLDER, BASE_FOLDER

existing_json_path = os.path.join(BASE_FOLDER, "prompt_experiments_augmentation.json")
backup_json_path = os.path.join(BASE_FOLDER, "prompt_experiments_augmentation_backup.json")

# Create backup copy first
shutil.copy2(existing_json_path, backup_json_path)
print(f" Created backup: {backup_json_path}")

all_entries = []

# Load existing entries and add folder path
with open(existing_json_path, "r") as f:
    for line in f:
        entry = json.loads(line)
        entry["folder"] = EXP_IMAGES_FOLDER  # Add folder path to existing entries
        all_entries.append(entry)

print(f" Loaded {len(all_entries)} existing entries with folder paths added.")

 Created backup: /hpc/group/youlab/ks723/storage/prompt_experiments_augmentation_backup.json
 Loaded 25700 existing entries with folder paths added.


In [ ]:
"""
Now we add the mutant library dataset images collected by Dongheon and Kristen for Jia
Note: the images have already been augmented first. (100x as usual)
"""

In [ ]:
# Step 2: Generate entries for mutant dataset
from itertools import combinations
import os
import natsort
from utils.local_config import MUTANT_EXP_FOLDER_AUG

mutant_folder = MUTANT_EXP_FOLDER_AUG

# Process mutant images with same logic as before
image_replicates_mutant = {}
for filename in natsort.natsorted(os.listdir(mutant_folder)):
    if filename.endswith('.TIF'):
        parts = filename.split('_')
        
    
        image_number = parts[0] 
        replicate_number = parts[1]
        rotation_angle = parts[2].replace('.TIF', '')

        
        key = (image_number, rotation_angle)
        
        if key not in image_replicates_mutant:
            image_replicates_mutant[key] = []
        image_replicates_mutant[key].append(replicate_number)

# Discard images with less than 2 replicates
image_replicates_mutant = {key: reps for key, reps in image_replicates_mutant.items() if len(reps) >= 2}

# Generate mutant entries with folder path
mutant_entries = []
for (image_number, rotation_angle), replicate_numbers in image_replicates_mutant.items():
    for rep1, rep2 in combinations(replicate_numbers, 2):
        src_filename = f"{image_number}_{rep1}_{rotation_angle}.TIF"
        tgt_filename = f"{image_number}_{rep2}_{rotation_angle}.TIF"
        
        entry = {
            "source": src_filename,
            "target": tgt_filename,
            "prompt": "",
            "folder": mutant_folder  # Add folder path for mutant entries
        }
        mutant_entries.append(entry)

print(f"Generated {len(mutant_entries)} mutant entries with folder paths.")

Generated 28500 mutant entries with folder paths.


In [5]:
# Step 3: Combine all entries and write to updated JSON file
combined_entries = all_entries + mutant_entries

# Write combined entries to original file
original_json_path = os.path.join(BASE_FOLDER, "prompt_experiments_augmentation.json")
with open(original_json_path, "w") as f:
    for entry in combined_entries:
        f.write(json.dumps(entry) + "\n")

print(f" Updated {original_json_path} with {len(combined_entries)} total entries.")
print(f"  - Original experimental images: {len(all_entries)}")
print(f"  - Mutant images: {len(mutant_entries)}")

 Updated /hpc/group/youlab/ks723/storage/prompt_experiments_augmentation.json with 54200 total entries.
  - Original experimental images: 25700
  - Mutant images: 28500


In [6]:
# Step 4: Verify all files exist
original_json_path = os.path.join(BASE_FOLDER, "prompt_experiments_augmentation.json")
with open(original_json_path, "r") as f:
    for line in f:
        entry = json.loads(line)
        src_path = os.path.join(entry["folder"], entry["source"])
        tgt_path = os.path.join(entry["folder"], entry["target"])
        assert os.path.exists(src_path), f"Source file {src_path} does not exist!"
        assert os.path.exists(tgt_path), f"Target file {tgt_path} does not exist!"

print("All source and target files verified to exist.")

All source and target files verified to exist.


In [ ]:
# Create a new mapping between experimental images with no augmentation
# this is a rough sanity check 
# Here we will do permuations too, i.e. for each pair (A,B) we will also have (B,A)
import os 
import json
from itertools import combinations
import natsort
from local_config import EXP_FOLDER_KS_NOAUG, MUTANT_EXP_FOLDER_NOAUG, BASE_FOLDER

folder_list= [EXP_FOLDER_KS_NOAUG, MUTANT_EXP_FOLDER_NOAUG]
total_entries = []
for folder in folder_list:
    
    # Process mutant images with same logic as before
    image_replicates = {}
    for filename in natsort.natsorted(os.listdir(folder)):
        if filename.endswith('.TIF'):
            parts = filename.split('_')
            
            if parts[0] == 'Fixed':
                image_number = parts[0] + '_' + parts[1]
                replicate_number = parts[2].replace('.TIF', '')
               
            else:
                image_number = parts[0]
                replicate_number = parts[1].replace('.TIF', '') 
                
        
            key = image_number
            
            if key not in image_replicates:
                image_replicates[key] = []
            image_replicates[key].append(replicate_number)

    # Discard images with less than 2 replicates
    image_replicates = {key: reps for key, reps in image_replicates.items() if len(reps) >= 2}

    # Generate mutant entries with folder path
    
    for image_number, replicate_numbers in image_replicates.items():
        for rep1, rep2 in combinations(replicate_numbers, 2):
            src_filename = f"{image_number}_{rep1}.TIF"
            tgt_filename = f"{image_number}_{rep2}.TIF"
            
            entry = {
                "source": src_filename,
                "target": tgt_filename,
                "prompt": "",
                "folder": folder  # Add folder path for mutant entries
            }
            total_entries.append(entry)
            # do reverse too
            src_filename = f"{image_number}_{rep2}.TIF"
            tgt_filename = f"{image_number}_{rep1}.TIF"
            
            entry = {
                "source": src_filename,
                "target": tgt_filename,
                "prompt": "",
                "folder": folder  # Add folder path for mutant entries
            }
            total_entries.append(entry)

    print(f"Generated {len(total_entries)} mutant entries with folder paths for folder {folder}.")


# Write to json file 

with open(os.path.join(BASE_FOLDER, "prompt_experiments_noaug_permutations.json"), "w") as f:
    for entry in total_entries:
        f.write(json.dumps(entry) + "\n")



Generated 514 mutant entries with folder paths for folder /hpc/group/youlab/ks723/storage/Exp_images/Final_folder_uniform_fixedseed_preprocess_noaug.
Generated 1084 mutant entries with folder paths for folder /hpc/group/youlab/ks723/storage/Exp_images/NL_evolution_library_Image_preprocess_noaug.


In [ ]:
# Create a new mapping between experimental images with no augmentation
# Here we will do permuations too, i.e. for each pair (A,B) we will also have (B,A)
import os 
import json
from itertools import combinations
import natsort
from utils.local_config import EXP_IMAGES_FOLDER, MUTANT_EXP_FOLDER_AUG
from utils.local_config import BASE_FOLDER
folder_list= [EXP_IMAGES_FOLDER, MUTANT_EXP_FOLDER_AUG]
total_entries = []
for folder in folder_list:
    
    # Process mutant images with same logic as before
    image_replicates = {}
    for filename in natsort.natsorted(os.listdir(folder)):
        if filename.endswith('.TIF'):
            parts = filename.split('_')
            
            if parts[0] == 'Fixed':
                image_number = parts[0] + '_' + parts[1]
                replicate_number = parts[2]
                rotation_angle = parts[3].replace('.TIF', '')  # e.g., "rot356.4" or "rot338.400000000003"
            else:
                image_number = parts[0]
                replicate_number = parts[1]
                rotation_angle = parts[2].replace('.TIF', '')  # e.g., "rot356.4" or "rot338.400000000003"
        
            # Group by image number AND rotation angle
            key = (image_number, rotation_angle)
            
            if key not in image_replicates:
                image_replicates[key] = []
            image_replicates[key].append(replicate_number)

    # Discard images with less than 2 replicates
    image_replicates = {key: reps for key, reps in image_replicates.items() if len(reps) >= 2}

    # Generate mutant entries with folder path
    
    for (image_number, rotation_angle), replicate_numbers in image_replicates.items():
        for rep1, rep2 in combinations(replicate_numbers, 2):
            src_filename = f"{image_number}_{rep1}_{rotation_angle}.TIF"
            tgt_filename = f"{image_number}_{rep2}_{rotation_angle}.TIF"
            
            entry = {
                "source": src_filename,
                "target": tgt_filename,
                "prompt": "",
                "folder": folder  # Add folder path for mutant entries
            }
            total_entries.append(entry)
            # do reverse too
            src_filename = f"{image_number}_{rep2}_{rotation_angle}.TIF"
            tgt_filename = f"{image_number}_{rep1}_{rotation_angle}.TIF"
            
            entry = {
                "source": src_filename,
                "target": tgt_filename,
                "prompt": "",
                "folder": folder  # Add folder path for mutant entries
            }
            total_entries.append(entry)

    print(f"Generated {len(total_entries)} mutant entries with folder paths for folder {folder}.")


# Write to json file 

with open(os.path.join(BASE_FOLDER, "prompt_experiments_permutations.json"), "w") as f:
    for entry in total_entries:
        f.write(json.dumps(entry) + "\n")

# TOO MUCH DATA NOW! 
# Generated 9189100 mutant entries with folder paths for folder /hpc/group/youlab/ks723/storage/Exp_images/Final_folder_uniform_fixedseed_100AUG.
# Generated 17710600 mutant entries with folder paths for folder /hpc/group/youlab/ks723/storage/Exp_images/NL_evolution_library_Image_AUG100.

Generated 51400 mutant entries with folder paths for folder /hpc/group/youlab/ks723/storage/Exp_images/Final_folder_uniform_fixedseed_100AUG.
Generated 108400 mutant entries with folder paths for folder /hpc/group/youlab/ks723/storage/Exp_images/NL_evolution_library_Image_AUG100.


In [13]:
# Step 4: Verify all files exist
original_json_path = os.path.join(BASE_FOLDER, "prompt_experiments_permutations.json")
with open(original_json_path, "r") as f:
    for line in f:
        entry = json.loads(line)
        src_path = os.path.join(entry["folder"], entry["source"])
        tgt_path = os.path.join(entry["folder"], entry["target"])
        assert os.path.exists(src_path), f"Source file {src_path} does not exist!"
        assert os.path.exists(tgt_path), f"Target file {tgt_path} does not exist!"

print("All source and target files verified to exist.")

All source and target files verified to exist.


In [ ]:
"""Adding Emrah's data from the PaKp paper
Note the images have been augmented 100x already, so we will just add them to the existing json file
"""

In [ ]:
# Now we add Emrah's data too, and seperate some of the original data into an explicity defined test set
# We also remove some of the original data from Nan's mutant library into an explicit test set(combined with KuiZhu's data)
# Create a new mapping between experimental images with no augmentation
# Here we will do permuations too, i.e. for each pair (A,B) we will also have (B,A)
import os 
import json
from itertools import combinations
import natsort
from utils.local_config import EXP_IMAGES_FOLDER, MUTANT_EXP_FOLDER_AUG_TRAINVALONLY, EMRAH_EXP_FOLDER_AUG
from utils.local_config import BASE_FOLDER

folder_list= [EXP_IMAGES_FOLDER, MUTANT_EXP_FOLDER_AUG_TRAINVALONLY, EMRAH_EXP_FOLDER_AUG]
total_entries = []
for folder in folder_list:
    entries_before = len(total_entries)  # Track count before processing this folder
    
    # Process mutant images with same logic as before
    image_replicates = {}
    for filename in natsort.natsorted(os.listdir(folder)):
        if filename.endswith('.TIF'):
            parts = filename.split('_')
            
            if parts[0] == 'Fixed':
                image_number = parts[0] + '_' + parts[1]
                replicate_number = parts[2]
                rotation_angle = parts[3].replace('.TIF', '')  # e.g., "rot356.4" or "rot338.400000000003"
            else:
                image_number = parts[0]
                replicate_number = parts[1]
                rotation_angle = parts[2].replace('.TIF', '')  # e.g., "rot356.4" or "rot338.400000000003"
        
            # Group by image number AND rotation angle
            key = (image_number, rotation_angle)
            
            if key not in image_replicates:
                image_replicates[key] = []
            image_replicates[key].append(replicate_number)

    # Discard images with less than 2 replicates
    image_replicates = {key: reps for key, reps in image_replicates.items() if len(reps) >= 2}

    # Generate mutant entries with folder path
    
    for (image_number, rotation_angle), replicate_numbers in image_replicates.items():
        for rep1, rep2 in combinations(replicate_numbers, 2):
            src_filename = f"{image_number}_{rep1}_{rotation_angle}.TIF"
            tgt_filename = f"{image_number}_{rep2}_{rotation_angle}.TIF"
            
            entry = {
                "source": src_filename,
                "target": tgt_filename,
                "prompt": "",
                "folder": folder  # Add folder path for mutant entries
            }
            total_entries.append(entry)
            # do reverse too
            src_filename = f"{image_number}_{rep2}_{rotation_angle}.TIF"
            tgt_filename = f"{image_number}_{rep1}_{rotation_angle}.TIF"
            
            entry = {
                "source": src_filename,
                "target": tgt_filename,
                "prompt": "",
                "folder": folder  # Add folder path for mutant entries
            }
            total_entries.append(entry)

    entries_added = len(total_entries) - entries_before
    print(f"Generated {entries_added} entries with folder paths for folder {folder}.")

print(f"\nTotal entries across all folders: {len(total_entries)}")

# Write to json file 

with open(os.path.join(BASE_FOLDER, "prompt_experiments_3datasets_permutations_resfix.json"), "w") as f:
    for entry in total_entries:
        f.write(json.dumps(entry) + "\n")


Generated 51400 entries with folder paths for folder /hpc/group/youlab/ks723/storage/Exp_images/Final_folder_uniform_fixedseed_100AUG.
Generated 50400 entries with folder paths for folder /hpc/group/youlab/ks723/storage/Exp_images/NL_evolution_library_Image_AUG100_TrainValOnlyAFTER_RES_2000_FIX.
Generated 198800 entries with folder paths for folder /hpc/group/youlab/ks723/storage/Exp_images/EmrahPaKp_dataset_renamed_AUG100_TrainValOnly.

Total entries across all folders: 300600


In [3]:
# Step 4: Verify all files exist
original_json_path = os.path.join(BASE_FOLDER, "prompt_experiments_3datasets_permutations_resfix.json")
with open(original_json_path, "r") as f:
    for line in f:
        entry = json.loads(line)
        src_path = os.path.join(entry["folder"], entry["source"])
        tgt_path = os.path.join(entry["folder"], entry["target"])
        assert os.path.exists(src_path), f"Source file {src_path} does not exist!"
        assert os.path.exists(tgt_path), f"Target file {tgt_path} does not exist!"

print("All source and target files verified to exist.")

All source and target files verified to exist.


In [4]:
# find length of json file 

with open(os.path.join(BASE_FOLDER, "prompt_experiments_3datasets_permutations.json"), "r") as f:
    line_count = sum(1 for line in f)
print(f"Number of entries: {line_count}")

Number of entries: 300600


In [ ]:
# create a balanced prompt json with equal number of entries from each dataset
# limitation: this might fail if we have less than the required number of images in any dataset, address later if necesary

import os 
import json 
import random

from utils.local_config import BASE_FOLDER

total_dataset_size= 100000  # total entries in the final json

# read json files, find unique dataset folders 
# then divide total size equally among them according to total size

json_path = os.path.join(BASE_FOLDER, f"prompt_experiments_3datasets_permutations.json")
dataset_folders = set()

entries = []

with open(json_path, "r") as f:
    for line in f:
        entry = json.loads(line)
        entries.append(entry)

        if entry["folder"] not in dataset_folders:
            dataset_folders.add(entry["folder"])

# figure out how many entries we have to include from each dataset
# equal to total_dataset_size / number of unique folders

images_per_dataset = total_dataset_size // len(dataset_folders)  # floor division, lower limit is number is not exactly divisible

# now create the balanced entries
# we will randomly choose images_per_dataset entries from each folder

random.seed(42)  # Set seed for reproducibility
balanced_entries = []
for folder in dataset_folders:
    folder_entries = [entry for entry in entries if entry["folder"] == folder]
    selected_entries = random.sample(folder_entries, images_per_dataset)
    balanced_entries.extend(selected_entries)
    
# write to new json file 

with open(os.path.join(BASE_FOLDER, f"prompt_experiments_3datasets_permutations_balanced_{total_dataset_size}.json"), "w") as f:
    for entry in balanced_entries:
        f.write(json.dumps(entry) + "\n")

# Step 4: Verify all files exist
original_json_path = os.path.join(BASE_FOLDER, f"prompt_experiments_3datasets_permutations_balanced_{total_dataset_size}.json")
with open(original_json_path, "r") as f:
    for line in f:
        entry = json.loads(line)
        src_path = os.path.join(entry["folder"], entry["source"])
        tgt_path = os.path.join(entry["folder"], entry["target"])
        assert os.path.exists(src_path), f"Source file {src_path} does not exist!"
        assert os.path.exists(tgt_path), f"Target file {tgt_path} does not exist!"
print("All source and target files verified to exist.")

print(f" Generated balanced json with {len(balanced_entries)} entries.")

All source and target files verified to exist.
 Generated balanced json with 99999 entries.


In [3]:
"""Adding Kristen's 2sp data from the PaKp paper
Note the images have been augmented 100x already
Here we will test with fine-tuning the model on this dataset, so we will create a new prompt json file
"""

# Now we add Kristen's data too
# Here we will do permuations too, i.e. for each pair (A,B) we will also have (B,A)
import os 
import json
from itertools import combinations
import natsort
# from utils.local_config import EXP_IMAGES_FOLDER, MUTANT_EXP_FOLDER_AUG_TRAINVALONLY, EMRAH_EXP_FOLDER_AUG
from utils.local_config import BASE_FOLDER, KRISTEN_EXP_FOLDER_2SP_FINALAUG

folder_list= [KRISTEN_EXP_FOLDER_2SP_FINALAUG]
total_entries = []
for folder in folder_list:
    entries_before = len(total_entries)  # Track count before processing this folder
    
    # Process mutant images with same logic as before
    image_replicates = {}
    for filename in natsort.natsorted(os.listdir(folder)):
        if filename.endswith('.jpg'):
            parts = filename.split('_')
            
            if parts[0] == 'Fixed':
                image_number = parts[0] + '_' + parts[1]
                replicate_number = parts[2]
                rotation_angle = parts[3].replace('.jpg', '')  # e.g., "rot356.4" or "rot338.400000000003"
            else:
                image_number = parts[0]
                replicate_number = parts[1]
                rotation_angle = parts[2].replace('.jpg', '')  # e.g., "rot356.4" or "rot338.400000000003"
        
            # Group by image number AND rotation angle
            key = (image_number, rotation_angle)
            
            if key not in image_replicates:
                image_replicates[key] = []
            image_replicates[key].append(replicate_number)

    # Discard images with less than 2 replicates
    image_replicates = {key: reps for key, reps in image_replicates.items() if len(reps) >= 2}

    # Generate mutant entries with folder path
    
    for (image_number, rotation_angle), replicate_numbers in image_replicates.items():
        for rep1, rep2 in combinations(replicate_numbers, 2):
            src_filename = f"{image_number}_{rep1}_{rotation_angle}.jpg"
            tgt_filename = f"{image_number}_{rep2}_{rotation_angle}.jpg"
            
            entry = {
                "source": src_filename,
                "target": tgt_filename,
                "prompt": "",
                "folder": folder  # Add folder path for mutant entries
            }
            total_entries.append(entry)
            # do reverse too
            src_filename = f"{image_number}_{rep2}_{rotation_angle}.jpg"
            tgt_filename = f"{image_number}_{rep1}_{rotation_angle}.jpg"
            
            entry = {
                "source": src_filename,
                "target": tgt_filename,
                "prompt": "",
                "folder": folder  # Add folder path for mutant entries
            }
            total_entries.append(entry)

    entries_added = len(total_entries) - entries_before
    print(f"Generated {entries_added} entries with folder paths for folder {folder}.")

print(f"\nTotal entries across all folders: {len(total_entries)}")

# Write to json file 

with open(os.path.join(BASE_FOLDER, "prompt_experiments_Kristen_2sp_finalaug_permutations.json"), "w") as f:
    for entry in total_entries:
        f.write(json.dumps(entry) + "\n")




Generated 109600 entries with folder paths for folder /hpc/group/youlab/ks723/storage/Exp_images/KL_automatedcrops_2species_filtered_renamed_AUG100.

Total entries across all folders: 109600
